In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
%env TENSORLY_BACKEND=cupy

env: CUPY_ACCELERATORS=cutensor,cub
env: TENSORLY_BACKEND=cupy


In [2]:
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: cupy


In [3]:
import cupy as cp
num_gpus = cp.cuda.runtime.getDeviceCount()
print(f"Available GPUs: {num_gpus}")

Available GPUs: 1


In [4]:
from moabb.datasets import *
from notebooks.data_loader import load_moabb_p300

epochs, labels, meta = load_moabb_p300(BNCI2014_008, subjects=[1], session=0)
epochs

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>



Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied


/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/paradigms/base.py:350: RuntimeWarning:

Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.



<EpochsArray | 4200 events (all good), 0 – 0.979 s (baseline off), ~12.3 MB, data loaded, with metadata,
 'Target': 700
 'NonTarget': 3500>

In [5]:
from notebooks.data_loader import postprocess

X,y = postprocess(epochs, labels)
X.shape

(4200, 8, 48)

In [6]:
from hoda.classification import BTTDACV

bttda = BTTDACV(
    verbose=True,
    n_jobs=1,
    extra_train_info=False,
    hoda_params=dict(
        tol=1e-6,
        max_iter=128,
        refit_shrinkage=False,
        verbose=True,
        extra_train_info=False
    )
)

In [7]:
from hoda.hoda import BTTDA, HODA, obj_tr
from hoda.cov import mode_scatter, ledoit_wolf_shrinkage

%load_ext line_profiler
%prun -s time bttda.fit(X,y)

fold=0, theta=0.0
Fitting block 1/16...


Forward model :  33%|███▎      | 42/127 [00:00<00:00, 338.74it/s]


Fitting block 2/16...


Forward model :   3%|▎         | 4/127 [00:00<00:00, 382.67it/s]


Fitting block 3/16...


...

Fitting block 14/15...


Forward model :   6%|▌         | 7/127 [00:00<00:00, 419.84it/s]


Fitting block 15/15...


Forward model :  10%|█         | 13/127 [00:00<00:00, 466.01it/s]


         60394949 function calls (59079005 primitive calls) in 196.194 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    77060   17.627    0.000   17.627    0.000 {built-in method cupy_backends.cuda.libs.cusolver.xsyevd}
    80400   13.941    0.000   35.880    0.000 cov.py:17(mode_scatter)
      835   13.523    0.016  131.117    0.157 hoda.py:178(fit_backward)
   203315   10.274    0.000   10.893    0.000 {built-in method cupy._core._routines_linalg.tensordot_core}
      815    9.492    0.012    9.562    0.012 _basic.py:1101(lstsq)
   175840    8.279    0.000    8.468    0.000 {method 'trace' of 'cupy._core.core._ndarray_base' objects}
    37695    7.559    0.000   11.182    0.000 hoda.py:43(obj_tr)
   115490    6.751    0.000   10.160    0.000 core.py:689(norm)
2427506/1141351    6.604    0.000   71.883    0.000 __init__.py:202(wrapped_backend_method)
    75390    5.334    0.000    8.773    0.000 util.py:147(flip_signs)


In [8]:
193.483

193.483